#### Importe as bibliotecas

In [6]:
# Configuração inicial
import pandas as pd
import numpy as np
from scipy import stats as st
import seaborn as sns
from matplotlib import pyplot as plt

#### Definição das configurações de visualização.

In [7]:
# Variável Global para aleatoriedade
SEED = 42

# Ajuste das quantidade das colunas exibidas
pd.set_option('display.max_columns', 100)

# Estilo e tamanho padrão dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

#### Funções Auxiliares

In [ ]:
def calcular_bins(dados):
    """
    Calcula a quantidade de bins (intervalos) recomendada para um histograma
    usando diferentes regras práticas de estatística descritiva.

    Parâmetros
    ----------
    dados : array-like (list, numpy.ndarray, pandas.Series)
        Conjunto de dados numéricos que será analisado.

    Retorna
    -------
    dict
        Dicionário contendo a quantidade de bins sugerida por três métodos:
        - "sqrt_rule": Regra da raiz quadrada (simples, rápida, baseada em √n)
        - "sturges_rule": Regra de Sturges (log2(n) + 1, boa para distribuições normais)
        - "freedman_diaconis": Regra de Freedman–Diaconis (baseada no IQR, mais robusta a outliers)

    Observações
    -----------
    - √n tende a funcionar bem em distribuições grandes e variadas.
    - Sturges é mais conservadora, útil para distribuições próximas à normal.
    - Freedman–Diaconis se adapta melhor a dados assimétricos e com outliers.

    Exemplo
    -------
    bins = calcular_bins(df['sales'])
    plt.hist(df['sales'], bins=bins['freedman_diaconis'])
    """

    # Calcula o tamanho da amostra (número de observações)
    n = len(dados)

    # Garante que os dados estão em formato NumPy array
    dados = np.asarray(dados)

    # 1. Raiz quadrada do tamanho da amostra
    bins_sqrt = int(np.sqrt(n))

    # 2. Regra de Sturges: log2(n) + 1
    bins_sturges = int(np.log2(n) + 1)

    # 3. Regra de Freedman–Diaconis
    q75, q25 = np.percentile(dados, [75, 25])   # Calcula os quartis
    iqr = q75 - q25                             # Intervalo interquartil (IQR)

    # Calcula a largura ideal do bin (bin_width) pela regra de FD
    bin_width = 2 * iqr * n ** (-1/3) if iqr > 0 else 1

    # Calcula o número de bins como o intervalo total / largura
    bins_fd = int((dados.max() - dados.min()) / bin_width) if bin_width > 0 else bins_sturges

    # Retorna os resultados como dicionário
    return {
        "sqrt_rule": bins_sqrt,
        "sturges_rule": bins_sturges,
        "freedman_diaconis": bins_fd
    }


In [ ]:
def outliers_iqr(series):
    """
    Identifica outliers em uma série numérica utilizando o método do Intervalo Interquartil (IQR).

    Parâmetros
    ----------
    series : pandas.Series
        Série numérica (coluna do DataFrame) onde serão identificados os outliers.

    Retorna
    -------
    mask : pandas.Series (boolean)
        Máscara booleana que indica True para os valores considerados outliers.
    lower : float
        Limite inferior para considerar um valor como não outlier.
    upper : float
        Limite superior para considerar um valor como não outlier.

    Exemplo
    -------
    mask, lower, upper = outliers_iqr(df['sales'])
    df[mask]  # retorna apenas os registros considerados outliers
    """

    # Calcula o primeiro quartil (25%)
    q1 = series.quantile(0.25)

    # Calcula o terceiro quartil (75%)
    q3 = series.quantile(0.75)

    # Calcula o intervalo interquartil (IQR = Q3 - Q1)
    iqr = q3 - q1

    # Define o limite inferior (Q1 - 1.5 * IQR)
    lower = q1 - 1.5 * iqr

    # Define o limite superior (Q3 + 1.5 * IQR)
    upper = q3 + 1.5 * iqr

    # Cria uma máscara booleana que marca os valores abaixo do limite inferior ou acima do superior
    mask = (series < lower) | (series > upper)

    # Retorna a máscara e os limites calculados
    return mask, lower, upper


In [ ]:
def outliers_zscore(series):
    """
    Identifica outliers em uma série numérica utilizando o método do Z-score.

    Parâmetros
    ----------
    series : pandas.Series
        Série numérica (coluna do DataFrame) onde serão identificados os outliers.

    Retorna
    -------
    mask : pandas.Series (boolean)
        Máscara booleana que indica True para os valores considerados outliers.
        (valores cujo Z-score absoluto é maior que 3).
    z : pandas.Series (float)
        Série contendo o valor padronizado (Z-score) de cada elemento.

    Observações
    -----------
    - O Z-score mede quantos desvios padrão cada valor está distante da média.
    - Por convenção, valores com |Z| > 3 são considerados outliers.
    - É utilizado ddof=0 para calcular o desvio padrão populacional (consistente com o Z-score).

    Exemplo
    -------
    mask, z = outliers_zscore(df['sales'])
    df[mask]  # retorna apenas os registros considerados outliers
    """

    # Remove valores ausentes (NaN) para cálculo da média e do desvio padrão
    s = series.dropna()

    # Calcula a média da série
    mean = s.mean()

    # Calcula o desvio padrão populacional (ddof=0)
    std = s.std(ddof=0)  # se usássemos ddof=1, seria o desvio padrão amostral

    # Calcula o Z-score: (valor - média) / desvio padrão
    # Se o desvio padrão for 0, usa 1.0 para evitar divisão por zero
    z = (series - mean) / (std if std != 0 else 1.0)

    # Cria uma máscara booleana: True para valores cujo |Z| > 3 (outliers)
    mask = z.abs() > 3

    # Retorna a máscara e a série de Z-scores
    return mask, z


In [ ]:
def safe_mode(series):
    """
    Calcula a moda (valor mais frequente) de uma série numérica ou categórica de forma segura.

    Parâmetros
    ----------
    series : pandas.Series
        Coluna de um DataFrame (numérica ou categórica) para a qual se deseja calcular a moda.

    Retorna
    -------
    valor : qualquer tipo ou float
        Retorna o valor mais frequente (moda) da série.
        Caso não exista uma moda única (ex.: múltiplos valores empatados),
        retorna NaN para evitar erro.

    Observações
    -----------
    - Usa a função mode da biblioteca `statistics`.
    - O método `.dropna()` é aplicado para ignorar valores ausentes (NaN).
    - Em cenários com múltiplas modas, `statistics.mode()` gera `StatisticsError`.
      Esse erro é tratado e a função retorna `np.nan`.

    Exemplo
    -------
    >>> safe_mode(pd.Series([1, 2, 2, 3, 4]))
    2

    >>> safe_mode(pd.Series([1, 1, 2, 2]))  # múltiplas modas
    nan
    """

    # Remove valores ausentes antes do cálculo da moda
    try:
        return mode(series.dropna())
    except StatisticsError:
        # Se houver empate (múltiplas modas), retorna NaN
        return np.nan


#### Leitura dos dados

In [8]:
df = pd.read_csv('games.csv')
df.head()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
